In [39]:
from pathlib import Path
import html
import warnings

import joblib
import numpy as np
import pandas as pd
import ipywidgets as widgets
import plotly.graph_objects as go

from IPython.display import display, HTML
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=FutureWarning)

In [40]:
DATA_PATH = Path("data_visualization.csv")
MODEL_PATH = Path("publisher_clustering_model.joblib")

DEFAULT_N_CLUSTERS = 4
RANDOM_STATE = 42
PLOT_HEIGHT_1 = 500
PLOT_HEIGHT_2 = 520

In [41]:
DASHBOARD_CSS = """
<style>
.dashboard-shell {
    background: #0b0f14;
    color: #f4f7fb;
    padding: 22px;
    border-radius: 18px;
    border: 1px solid #1f2937;
    font-family: Inter, Arial, sans-serif;
    box-shadow: 0 8px 24px rgba(0,0,0,0.25);
    margin-top: 10px;
}

.dashboard-title {
    font-size: 30px;
    font-weight: 700;
    letter-spacing: 0.4px;
    margin-bottom: 4px;
    color: #f9fafb;
}

.dashboard-subtitle {
    font-size: 14px;
    color: #9ca3af;
    margin-bottom: 8px;
}

.section-title {
    font-size: 18px;
    font-weight: 700;
    color: #f9fafb;
    margin-top: 18px;
    margin-bottom: 10px;
}

.metric-grid {
    display: grid;
    grid-template-columns: repeat(4, minmax(170px, 1fr));
    gap: 12px;
    margin-bottom: 16px;
}

.metric-card {
    background: linear-gradient(180deg, #111827 0%, #0f172a 100%);
    border: 1px solid #1f2937;
    border-radius: 14px;
    padding: 14px 16px;
    box-shadow: 0 6px 18px rgba(0,0,0,0.20);
}

.metric-label {
    font-size: 12px;
    color: #9ca3af;
    margin-bottom: 8px;
    text-transform: uppercase;
    letter-spacing: 0.5px;
}

.metric-value {
    font-size: 22px;
    font-weight: 700;
    color: #f9fafb;
}

.metric-note {
    font-size: 12px;
    color: #34d399;
    margin-top: 6px;
}

.summary-box {
    background: linear-gradient(180deg, #111827 0%, #0f172a 100%);
    border: 1px solid #1f2937;
    border-radius: 14px;
    padding: 16px;
    margin-bottom: 16px;
}

.summary-title {
    font-size: 16px;
    font-weight: 700;
    color: #f9fafb;
    margin-bottom: 8px;
}

.summary-text {
    font-size: 14px;
    color: #d1d5db;
    line-height: 1.6;
}

.table-wrap {
    max-height: 420px;
    overflow-y: auto;
    border: 1px solid #1f2937;
    border-radius: 12px;
    margin-bottom: 16px;
}

table.dark-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
    background: #0f172a;
    color: #f3f4f6;
}

table.dark-table th,
table.dark-table td {
    border-bottom: 1px solid #1f2937;
    padding: 10px 12px;
    text-align: left;
}

table.dark-table th {
    background: #111827;
    color: #e5e7eb;
    position: sticky;
    top: 0;
    z-index: 1;
}

table.dark-table tr:hover td {
    background: #111827;
}

.small-note {
    font-size: 12px;
    color: #9ca3af;
    margin-top: 4px;
    margin-bottom: 12px;
}

.control-box {
    background: #111827;
    border: 1px solid #1f2937;
    border-radius: 14px;
    padding: 14px;
    margin: 10px 0 16px 0;
}

.jp-OutputArea-output,
.jupyter-widgets-output-area {
    background: transparent !important;
}
</style>
"""
display(HTML(DASHBOARD_CSS))

In [42]:
def load_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Could not find {path}")

    df = pd.read_csv(path)

    df = df.dropna(subset=["ticker", "publisher"]).copy()
    df["publisher"] = df["publisher"].astype(str).str.strip()
    df["ticker"] = df["ticker"].astype(str).str.strip()

    numeric_cols = [
        "sentiment_encoded",
        "return_30d",
        "return_6m",
        "long_term",
        "accuracy",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    fill_defaults = {
        "accuracy": 0.0,
        "sentiment_encoded": 0.0,
        "long_term": 0.0,
        "return_30d": 0.0,
        "return_6m": 0.0,
    }

    for col, val in fill_defaults.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)

    if "headline" not in df.columns:
        df["headline"] = ""

    return df


data = load_data(DATA_PATH)

In [43]:
def build_aggregates(df: pd.DataFrame):
    ticker_features = df.groupby("ticker").agg(
        avg_sentiment=("sentiment_encoded", "mean"),
        sentiment_std=("sentiment_encoded", "std"),
        bullish_rate=("sentiment_encoded", lambda x: (x == 1).mean()),
        bearish_rate=("sentiment_encoded", lambda x: (x == -1).mean()),
        neutral_rate=("sentiment_encoded", lambda x: (x == 0).mean()),
        avg_return_30d=("return_30d", "mean"),
        avg_return_6m=("return_6m", "mean"),
        vol_30d=("return_30d", "std"),
        article_count=("headline", "count"),
        long_term_ratio=("long_term", "mean"),
    ).reset_index()

    ticker_features["sentiment_std"] = ticker_features["sentiment_std"].fillna(0)
    ticker_features["vol_30d"] = ticker_features["vol_30d"].fillna(0)
    ticker_features["log_article_count"] = np.log1p(ticker_features["article_count"])

    publisher_overall = df.groupby("publisher").agg(
        overall_articles=("headline", "count"),
        overall_accuracy=("accuracy", "mean"),
        overall_avg_sentiment=("sentiment_encoded", "mean"),
        overall_long_term_ratio=("long_term", "mean"),
        unique_tickers=("ticker", "nunique"),
    ).reset_index()

    publisher_overall["log_overall_articles"] = np.log1p(publisher_overall["overall_articles"])

    publisher_by_ticker = df.groupby(["ticker", "publisher"]).agg(
        ticker_articles=("headline", "count"),
        ticker_accuracy=("accuracy", "mean"),
        ticker_avg_sentiment=("sentiment_encoded", "mean"),
        ticker_long_term_ratio=("long_term", "mean"),
    ).reset_index()

    publisher_lookup = publisher_by_ticker.merge(
        publisher_overall,
        on="publisher",
        how="left"
    )

    return ticker_features, publisher_overall, publisher_lookup


ticker_features, publisher_overall, publisher_lookup = build_aggregates(data)

PUBLISHER_CLUSTER_COLS = [
    "overall_accuracy",
    "overall_avg_sentiment",
    "log_overall_articles",
    "unique_tickers",
    "overall_long_term_ratio",
]

In [44]:
def load_or_build_cluster_bundle(
    publisher_overall,
    model_path,
    feature_cols,
    n_clusters=DEFAULT_N_CLUSTERS,
    random_state=RANDOM_STATE,
):
    X = publisher_overall[feature_cols].fillna(0)

    if model_path.exists():
        bundle = joblib.load(model_path)

        required_keys = {"scaler", "kmeans", "pca", "publisher_cluster_cols"}
        missing = required_keys - set(bundle.keys())
        if missing:
            raise ValueError(f"Model bundle missing keys: {missing}")

        if list(bundle["publisher_cluster_cols"]) != list(feature_cols):
            raise ValueError(
                "Saved model publisher_cluster_cols do not match dashboard feature order."
            )

        scaler = bundle["scaler"]
        kmeans = bundle["kmeans"]
        pca = bundle["pca"]

    else:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        kmeans = KMeans(
            n_clusters=n_clusters,
            random_state=random_state,
            n_init=20
        )
        kmeans.fit(X_scaled)

        pca = PCA(n_components=2, random_state=random_state)
        pca.fit(X_scaled)

        bundle = {
            "publisher_cluster_cols": list(feature_cols),
            "scaler": scaler,
            "kmeans": kmeans,
            "pca": pca,
            "silhouette_score": None,
            "n_clusters": kmeans.n_clusters,
        }
        joblib.dump(bundle, model_path)

    X_scaled = scaler.transform(X)
    clusters = kmeans.predict(X_scaled)
    pca_vals = pca.transform(X_scaled)

    publisher_overall = publisher_overall.copy()
    publisher_overall["cluster"] = clusters
    publisher_overall["pca1"] = pca_vals[:, 0]
    publisher_overall["pca2"] = pca_vals[:, 1]

    return publisher_overall, bundle

publisher_overall, model_bundle = load_or_build_cluster_bundle(
    publisher_overall=publisher_overall,
    model_path=MODEL_PATH,
    feature_cols=PUBLISHER_CLUSTER_COLS,
)

In [45]:
publisher_lookup = publisher_lookup.drop(columns=["cluster", "pca1", "pca2"], errors="ignore").merge(
    publisher_overall[["publisher", "cluster", "pca1", "pca2"]],
    on="publisher",
    how="left"
)

cluster_summary = publisher_overall.groupby("cluster").agg(
    overall_accuracy=("overall_accuracy", "mean"),
    overall_articles=("overall_articles", "mean"),
    overall_avg_sentiment=("overall_avg_sentiment", "mean"),
    unique_tickers=("unique_tickers", "mean"),
).reset_index()

accuracy_median = publisher_overall["overall_accuracy"].median()
articles_median = publisher_overall["overall_articles"].median()
sentiment_median = publisher_overall["overall_avg_sentiment"].median()
coverage_median = publisher_overall["unique_tickers"].median()

cluster_descriptions = {}
for _, row in cluster_summary.iterrows():
    cluster_id = int(row["cluster"])
    parts = []

    parts.append("higher accuracy" if row["overall_accuracy"] >= accuracy_median else "lower accuracy")
    parts.append("high volume" if row["overall_articles"] >= articles_median else "lower volume")
    parts.append("broad coverage" if row["unique_tickers"] >= coverage_median else "more specialized coverage")
    parts.append("more positive sentiment" if row["overall_avg_sentiment"] >= sentiment_median else "more neutral/negative sentiment")

    cluster_descriptions[cluster_id] = ", ".join(parts).capitalize() + "."

ticker_to_rows = {
    ticker: df.reset_index(drop=True)
    for ticker, df in publisher_lookup.groupby("ticker", sort=False)
}

ticker_feature_map = ticker_features.set_index("ticker").to_dict("index")
publisher_overall_map = publisher_overall.set_index("publisher").to_dict("index")
global_stats_base = publisher_overall.copy()

In [46]:
def fmt3(x):
    return "N/A" if pd.isna(x) else f"{x:.3f}"

def make_metric_card(label, value, note=""):
    return f"""
    <div class="metric-card">
        <div class="metric-label">{html.escape(str(label))}</div>
        <div class="metric-value">{html.escape(str(value))}</div>
        <div class="metric-note">{html.escape(str(note))}</div>
    </div>
    """

def df_to_dark_html(df: pd.DataFrame) -> str:
    return f'<div class="table-wrap">{df.to_html(index=False, classes="dark-table", escape=False)}</div>'

In [47]:
def get_filtered_ticker_df(selected_ticker: str, min_articles: int) -> pd.DataFrame:
    df = ticker_to_rows.get(selected_ticker, pd.DataFrame()).copy()
    if df.empty:
        return df
    return df[df["ticker_articles"] >= min_articles].sort_values("ticker_articles", ascending=False).reset_index(drop=True)

def build_summary_html(selected_ticker: str, selected_publisher: str, min_articles: int) -> str:
    ticker_row = ticker_feature_map[selected_ticker]
    ticker_publishers = get_filtered_ticker_df(selected_ticker, min_articles)

    n_publishers = len(ticker_publishers)
    top_accuracy = ticker_publishers["ticker_accuracy"].max() if not ticker_publishers.empty else np.nan

    cards_html = f"""
    <div class="metric-grid">
        {make_metric_card("Selected Ticker", selected_ticker, "Current dashboard focus")}
        {make_metric_card("Ticker Articles", int(ticker_row["article_count"]), "All articles in dataset")}
        {make_metric_card("Avg Sentiment", fmt3(ticker_row["avg_sentiment"]), "Mean sentiment for selected ticker")}
        {make_metric_card("Avg 30d Return", fmt3(ticker_row["avg_return_30d"]), "Mean 30-day return")}
        {make_metric_card("Avg 6m Return", fmt3(ticker_row["avg_return_6m"]), "Mean 6-month return")}
        {make_metric_card("30d Volatility", fmt3(ticker_row["vol_30d"]), "Std. dev. of 30-day return")}
        {make_metric_card("Publishers Shown", int(n_publishers), f"Filtered at ≥ {min_articles} article(s)")}
        {make_metric_card("Best Ticker Accuracy", fmt3(top_accuracy), "Top publisher for this ticker")}
    </div>
    """

    if selected_publisher == "All Publishers":
        return cards_html

    overall_row = publisher_overall_map.get(selected_publisher)
    if overall_row is None:
        return cards_html

    ticker_row_pub = ticker_publishers[ticker_publishers["publisher"] == selected_publisher]
    cluster_id = int(overall_row["cluster"])
    cluster_text = cluster_descriptions.get(cluster_id, "No description available.")

    if not ticker_row_pub.empty:
        ticker_row_pub = ticker_row_pub.iloc[0]
        ticker_specific_text = (
            f"For <b>{html.escape(selected_ticker)}</b>, this publisher has "
            f"<b>{int(ticker_row_pub['ticker_articles'])}</b> article(s), "
            f"<b>{ticker_row_pub['ticker_accuracy']:.3f}</b> ticker accuracy, and "
            f"<b>{ticker_row_pub['ticker_avg_sentiment']:.3f}</b> average ticker sentiment."
        )
    else:
        ticker_specific_text = (
            f"This publisher does not have rows for <b>{html.escape(selected_ticker)}</b> "
            f"under the current filter."
        )

    publisher_summary_html = f"""
    <div class="summary-box">
        <div class="summary-title">Selected Publisher Summary</div>
        <div class="summary-text">
            <b>{html.escape(selected_publisher)}</b> belongs to publisher cluster <b>{cluster_id}</b>.<br><br>
            Cluster profile: {html.escape(cluster_text)}<br><br>
            Overall accuracy: <b>{overall_row['overall_accuracy']:.3f}</b><br>
            Overall average sentiment: <b>{overall_row['overall_avg_sentiment']:.3f}</b><br>
            Overall articles: <b>{int(overall_row['overall_articles'])}</b><br>
            Unique tickers covered: <b>{int(overall_row['unique_tickers'])}</b><br><br>
            {ticker_specific_text}
        </div>
    </div>
    """

    return cards_html + publisher_summary_html


def build_accuracy_table(selected_ticker: str, min_articles: int, sort_by: str, ascending: bool) -> pd.DataFrame:
    df = get_filtered_ticker_df(selected_ticker, min_articles)

    if df.empty:
        return pd.DataFrame({"Message": ["No publisher rows available for this filter."]})

    table_df = df[[
        "publisher",
        "ticker_articles",
        "overall_articles",
        "overall_accuracy",
        "ticker_accuracy",
    ]].copy()

    table_df = table_df.rename(columns={
        "publisher": "Publisher",
        "ticker_articles": "Articles on Ticker",
        "overall_articles": "Overall Articles",
        "overall_accuracy": "Overall Accuracy",
        "ticker_accuracy": "Ticker Accuracy",
    })

    table_df = table_df.sort_values(sort_by, ascending=ascending).reset_index(drop=True)

    for col in ["Overall Accuracy", "Ticker Accuracy"]:
        table_df[col] = table_df[col].map(lambda x: f"{x:.3f}")

    return table_df


def build_global_stats_table(sort_by: str, ascending: bool) -> pd.DataFrame:
    table_df = global_stats_base[[
        "publisher",
        "cluster",
        "overall_articles",
        "unique_tickers",
        "overall_accuracy",
        "overall_avg_sentiment",
        "overall_long_term_ratio",
    ]].copy()

    table_df = table_df.rename(columns={
        "publisher": "Publisher",
        "cluster": "Cluster",
        "overall_articles": "Overall Articles",
        "unique_tickers": "Unique Tickers",
        "overall_accuracy": "Overall Accuracy",
        "overall_avg_sentiment": "Overall Avg Sentiment",
        "overall_long_term_ratio": "Long-Term Ratio",
    })

    table_df = table_df.sort_values(sort_by, ascending=ascending).reset_index(drop=True)

    for col in ["Overall Accuracy", "Overall Avg Sentiment", "Long-Term Ratio"]:
        table_df[col] = table_df[col].map(lambda x: f"{x:.3f}")

    return table_df

In [48]:
ticker_dropdown = widgets.Dropdown(
    options=sorted(ticker_features["ticker"].unique()),
    value=sorted(ticker_features["ticker"].unique())[0],
    description="Ticker:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="220px")
)

publisher_dropdown = widgets.Dropdown(
    options=["All Publishers"] + sorted(publisher_overall["publisher"].unique()),
    value="All Publishers",
    description="Publisher:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="360px")
)

min_articles_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=max(1, int(publisher_lookup["ticker_articles"].max())),
    step=1,
    description="Min Articles:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

accuracy_sort_dropdown = widgets.Dropdown(
    options=[
        "Articles on Ticker",
        "Overall Articles",
        "Overall Accuracy",
        "Ticker Accuracy",
    ],
    value="Articles on Ticker",
    description="Accuracy Table Sort:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px")
)

accuracy_sort_order = widgets.ToggleButtons(
    options=[("Desc", False), ("Asc", True)],
    value=False,
    description="Order:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="220px")
)

global_sort_dropdown = widgets.Dropdown(
    options=[
        "Overall Accuracy",
        "Overall Articles",
        "Unique Tickers",
        "Overall Avg Sentiment",
        "Long-Term Ratio",
        "Cluster",
    ],
    value="Overall Accuracy",
    description="Global Table Sort:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px")
)

global_sort_order = widgets.ToggleButtons(
    options=[("Desc", False), ("Asc", True)],
    value=False,
    description="Order:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="220px")
)

In [49]:
summary_html = widgets.HTML()
scatter_note = widgets.HTML('<div class="small-note">Each point is a publisher covering the selected ticker. Bubble size/color reflects article count on that ticker.</div>')
cluster_note = widgets.HTML('<div class="small-note">Publishers are grouped using overall accuracy, sentiment, article volume, ticker breadth, and long-term ratio.</div>')
accuracy_table_html = widgets.HTML()
global_table_html = widgets.HTML()

In [50]:
scatter_fig = go.FigureWidget()
cluster_fig = go.FigureWidget()

scatter_fig.update_layout(
    template="plotly_dark",
    height=PLOT_HEIGHT_1,
    paper_bgcolor="#0b0f14",
    plot_bgcolor="#111827",
    font=dict(color="#f9fafb"),
    xaxis=dict(title="Average Sentiment on Ticker", gridcolor="#253047", zerolinecolor="#334155"),
    yaxis=dict(title="Ticker Accuracy", gridcolor="#253047", zerolinecolor="#334155", range=[-0.02, 1.02]),
    margin=dict(l=50, r=40, t=60, b=50),
)

cluster_fig.update_layout(
    template="plotly_dark",
    height=PLOT_HEIGHT_2,
    paper_bgcolor="#0b0f14",
    plot_bgcolor="#111827",
    font=dict(color="#f9fafb"),
    xaxis=dict(title="PCA Component 1", gridcolor="#253047"),
    yaxis=dict(title="PCA Component 2", gridcolor="#253047"),
    margin=dict(l=50, r=40, t=60, b=50),
)

FigureWidget({
    'data': [],
    'layout': {'font': {'color': '#f9fafb'},
               'height': 520,
               'margin': {'b': 50, 'l': 50, 'r': 40, 't': 60},
               'paper_bgcolor': '#0b0f14',
               'plot_bgcolor': '#111827',
               'template': '...',
               'xaxis': {'gridcolor': '#253047', 'title': {'text': 'PCA Component 1'}},
               'yaxis': {'gridcolor': '#253047', 'title': {'text': 'PCA Component 2'}}}
})

In [51]:
def update_scatter_figure(selected_ticker: str, selected_publisher: str, min_articles: int):
    plot_df = get_filtered_ticker_df(selected_ticker, min_articles)

    with scatter_fig.batch_update():
        scatter_fig.data = []

        if plot_df.empty:
            scatter_fig.update_layout(
                title="Publisher Accuracy vs Sentiment",
                annotations=[dict(
                    text="No publishers match the current filter.",
                    x=0.5, y=0.5,
                    xref="paper", yref="paper",
                    showarrow=False,
                    font=dict(size=16)
                )]
            )
            return

        scatter_fig.layout.annotations = ()

        sizes = (
            np.interp(
                plot_df["ticker_articles"],
                (plot_df["ticker_articles"].min(), plot_df["ticker_articles"].max()),
                (14, 40)
            )
            if plot_df["ticker_articles"].nunique() > 1
            else np.full(len(plot_df), 24)
        )

        scatter_fig.add_scatter(
            x=plot_df["ticker_avg_sentiment"],
            y=plot_df["ticker_accuracy"],
            mode="markers",
            marker=dict(
                size=sizes,
                color=plot_df["ticker_articles"],
                colorscale="Viridis",
                showscale=True,
                colorbar=dict(title="Articles"),
                line=dict(width=1, color="#d1d5db"),
                opacity=0.92
            ),
            text=plot_df["publisher"],
            customdata=np.stack([
                plot_df["publisher"],
                plot_df["ticker_articles"],
                plot_df["overall_articles"],
                plot_df["overall_accuracy"],
                plot_df["ticker_accuracy"],
            ], axis=1),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Avg Sentiment on Ticker: %{x:.3f}<br>"
                "Ticker Accuracy: %{y:.3f}<br>"
                "Articles on Ticker: %{customdata[1]}<br>"
                "Overall Articles: %{customdata[2]}<br>"
                "Overall Accuracy: %{customdata[3]:.3f}<extra></extra>"
            ),
            name="Publishers"
        )

        if selected_publisher != "All Publishers":
            selected_row = plot_df[plot_df["publisher"] == selected_publisher]
            if not selected_row.empty:
                scatter_fig.add_scatter(
                    x=selected_row["ticker_avg_sentiment"],
                    y=selected_row["ticker_accuracy"],
                    mode="markers+text",
                    text=selected_row["publisher"],
                    textposition="top center",
                    marker=dict(
                        size=28,
                        symbol="diamond",
                        color="#f59e0b",
                        line=dict(width=2, color="white")
                    ),
                    hoverinfo="skip",
                    name="Selected Publisher"
                )

        scatter_fig.update_layout(
            title=f"Selected Ticker: {selected_ticker} | Publisher Accuracy vs Sentiment"
        )

        def handle_click(trace, points, state):
            if points.point_inds:
                clicked_publisher = plot_df.iloc[points.point_inds[0]]["publisher"]
                if publisher_dropdown.value != clicked_publisher:
                    publisher_dropdown.value = clicked_publisher

        scatter_fig.data[0].on_click(handle_click)


def update_cluster_figure(selected_ticker: str, selected_publisher: str):
    plot_df = publisher_overall.copy()
    highlight_publishers = set(
        publisher_lookup[publisher_lookup["ticker"] == selected_ticker]["publisher"].unique()
    )
    plot_df["in_selected_ticker"] = plot_df["publisher"].isin(highlight_publishers)

    with cluster_fig.batch_update():
        cluster_fig.data = []
        cluster_fig.layout.annotations = ()

        sizes = (
            np.interp(
                plot_df["overall_articles"],
                (plot_df["overall_articles"].min(), plot_df["overall_articles"].max()),
                (10, 32)
            )
            if plot_df["overall_articles"].nunique() > 1
            else np.full(len(plot_df), 18)
        )

        cluster_fig.add_scatter(
            x=plot_df["pca1"],
            y=plot_df["pca2"],
            mode="markers",
            marker=dict(
                size=sizes,
                color=plot_df["cluster"],
                colorscale="Turbo",
                showscale=True,
                colorbar=dict(title="Cluster"),
                line=dict(
                    width=np.where(plot_df["in_selected_ticker"], 2.5, 0.8),
                    color=np.where(plot_df["in_selected_ticker"], "#f8fafc", "#475569")
                ),
                opacity=0.9
            ),
            customdata=np.stack([
                plot_df["publisher"],
                plot_df["overall_accuracy"],
                plot_df["overall_articles"],
                plot_df["overall_avg_sentiment"],
                plot_df["cluster"],
            ], axis=1),
            text=plot_df["publisher"],
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Cluster: %{customdata[4]}<br>"
                "Overall Accuracy: %{customdata[1]:.3f}<br>"
                "Overall Articles: %{customdata[2]}<br>"
                "Overall Avg Sentiment: %{customdata[3]:.3f}<extra></extra>"
            ),
            name="Publishers"
        )

        if selected_publisher != "All Publishers":
            selected_row = plot_df[plot_df["publisher"] == selected_publisher]
            if not selected_row.empty:
                cluster_fig.add_scatter(
                    x=selected_row["pca1"],
                    y=selected_row["pca2"],
                    mode="markers+text",
                    text=selected_row["publisher"],
                    textposition="top center",
                    marker=dict(
                        size=30,
                        symbol="star",
                        color="#22c55e",
                        line=dict(width=2, color="white")
                    ),
                    hoverinfo="skip",
                    name="Selected Publisher"
                )

        cluster_fig.update_layout(
            title="Publisher PCA / Clustering"
        )

        def handle_click(trace, points, state):
            if points.point_inds:
                clicked_publisher = plot_df.iloc[points.point_inds[0]]["publisher"]
                if publisher_dropdown.value != clicked_publisher:
                    publisher_dropdown.value = clicked_publisher

        cluster_fig.data[0].on_click(handle_click)

In [52]:
def refresh_dashboard(change=None):
    selected_ticker = ticker_dropdown.value
    selected_publisher = publisher_dropdown.value
    min_articles = min_articles_slider.value

    summary_html.value = build_summary_html(selected_ticker, selected_publisher, min_articles)

    update_scatter_figure(selected_ticker, selected_publisher, min_articles)
    update_cluster_figure(selected_ticker, selected_publisher)

    accuracy_df = build_accuracy_table(
        selected_ticker=selected_ticker,
        min_articles=min_articles,
        sort_by=accuracy_sort_dropdown.value,
        ascending=accuracy_sort_order.value
    )

    global_df = build_global_stats_table(
        sort_by=global_sort_dropdown.value,
        ascending=global_sort_order.value
    )

    accuracy_table_html.value = (
        '<div class="section-title">Publisher Coverage + Accuracy</div>' +
        df_to_dark_html(accuracy_df)
    )

    global_table_html.value = (
        '<div class="section-title">Global Publisher Sentiment</div>' +
        df_to_dark_html(global_df)
    )

In [53]:
for w in [
    ticker_dropdown,
    publisher_dropdown,
    min_articles_slider,
    accuracy_sort_dropdown,
    accuracy_sort_order,
    global_sort_dropdown,
    global_sort_order,
]:
    w.observe(refresh_dashboard, names="value")

In [54]:
header_html = widgets.HTML("""
<div class="dashboard-shell">
    <div class="dashboard-title">Financial News Source Dashboard</div>
    <div class="dashboard-subtitle">
        Explore publisher sentiment, clustering, and ticker-specific accuracy.
        Click a point in either plot to update the Publisher dropdown.
    </div>
</div>
""")

top_controls = widgets.HBox(
    [ticker_dropdown, publisher_dropdown, min_articles_slider],
    layout=widgets.Layout(gap="14px", flex_flow="row wrap")
)

table_controls = widgets.HBox(
    [accuracy_sort_dropdown, accuracy_sort_order, global_sort_dropdown, global_sort_order],
    layout=widgets.Layout(gap="14px", flex_flow="row wrap")
)

control_box = widgets.VBox(
    [top_controls, table_controls],
    layout=widgets.Layout()
)
control_box.add_class("control-box")

app = widgets.VBox([
    header_html,
    control_box,
    widgets.HTML('<div class="section-title">Ticker / Publisher Summary</div>'),
    summary_html,
    widgets.HTML('<div class="section-title">Average Sentiment vs Ticker Accuracy by Publisher</div>'),
    scatter_note,
    scatter_fig,
    widgets.HTML('<div class="section-title">PCA / Clustering of Publishers</div>'),
    cluster_note,
    cluster_fig,
    accuracy_table_html,
    global_table_html,
])

refresh_dashboard()
display(app)